# Huấn Luyện Mô Hình Deep Learning CRNN-CTC Giải Mã CAPTCHA HUIT

> **Dự án:** Khóa luận Tốt nghiệp - Hệ thống Gợi ý Học tập Cá nhân hóa  
> **Mục tiêu:** Xây dựng mô hình OCR nhận diện CAPTCHA hình ảnh trên Cổng thông tin Sinh viên HUIT với độ chính xác cao (>95%) và thời gian suy luận siêu tốc (<10ms trên CPU) bằng ONNX Runtime.

---

### Kiến Trúc Hệ Thống
```
Ảnh CAPTCHA (1x40x120 Grayscale)
       │
       ▼
┌────────────────────────────────────────────────────────┐
│ 1. CNN Backbone (5 Blocks: Conv2D + BatchNorm + ReLU)  │
└────────────────────────────────────────────────────────┘
       │
       ▼ (Feature Sequence: 29 time-steps x 256 channels)
┌────────────────────────────────────────────────────────┐
│ 2. Sequence Modeling: 2-layer Bidirectional LSTM       │
└────────────────────────────────────────────────────────┘
       │
       ▼ (29 x 256)
┌────────────────────────────────────────────────────────┐
│ 3. Linear Head + LogSoftmax -> CTC Loss / Greedy Search │
└────────────────────────────────────────────────────────┘
       │
       ▼
Chuỗi CAPTCHA dự đoán (Ví dụ: "QNJU", "63L8")
```

## Phần 1: Cài Đặt Môi Trường và Import Thư Viện

In [ ]:
import os
import sys
import time
import json
import csv
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import onnx
import onnxruntime as ort

# Thiết lập hạt giống ngẫu nhiên để tái lập kết quả
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Thiết bị tính toán: {device}")
print(f"[*] PyTorch Version: {torch.__version__}")
print(f"[*] ONNX Runtime Version: {ort.__version__}")

## Phần 2: Đọc Dữ Liệu, Tiền Xử Lý & Lập Bảng Từ Vựng (Vocabulary)

In [ ]:
# Cấu hình đường dẫn dữ liệu
BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in locals() else os.getcwd()
DATA_DIR = os.path.abspath(os.path.join(BASE_DIR, "..", "data"))
SYNTHETIC_DIR = os.path.join(DATA_DIR, "captchas", "synthetic")
LABELS_CSV = os.path.join(SYNTHETIC_DIR, "labels.csv")

print(f"[*] Thư mục dữ liệu: {DATA_DIR}")
print(f"[*] File nhãn: {LABELS_CSV}")

# Đọc danh sách mẫu từ file labels.csv
all_samples = []
if os.path.exists(LABELS_CSV):
    with open(LABELS_CSV, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rel_path = row.get("filename", "")
            label = row.get("label", "").strip()
            full_path = os.path.join(SYNTHETIC_DIR, rel_path)
            if os.path.exists(full_path) and label:
                all_samples.append((full_path, label))

print(f"[*] Đã nạp thành công {len(all_samples)} mẫu CAPTCHA.")

# Xây dựng Bảng từ vựng (Vocabulary: 0 là CTC Blank)
unique_chars = sorted(list(set([c for _, label in all_samples for c in label])))
char2idx = {"<blank>": 0}
for idx, c in enumerate(unique_chars, start=1):
    char2idx[c] = idx

idx2char = {v: k for k, v in char2idx.items()}
num_classes = len(char2idx)

print(f"[*] Số lượng lớp phân loại (Num Classes): {num_classes}")
print(f"[*] Bảng ký tự: {''.join(unique_chars)}")

# Phân chia Train (80%) / Val (10%) / Test (10%)
random.shuffle(all_samples)
n_total = len(all_samples)
n_train = int(n_total * 0.8)
n_val = int(n_total * 0.1)

train_samples = all_samples[:n_train]
val_samples = all_samples[n_train:n_train + n_val]
test_samples = all_samples[n_train + n_val:]

print(f"[*] Phân chia tập dữ liệu: Train={len(train_samples)}, Val={len(val_samples)}, Test={len(test_samples)}")

## Phần 3: Định Nghĩa PyTorch Dataset & Collate Function

In [ ]:
class CaptchaDataset(Dataset):
    """
    Dataset tùy biến cho ảnh CAPTCHA:
    - Đọc ảnh qua PIL -> Chuyển Grayscale ('L')
    - Resize về kích thước chuẩn (120 x 40)
    - Chuẩn hóa giá trị pixel về dải [0.0, 1.0]
    """
    def __init__(self, samples: list, char2idx: dict, img_width: int = 120, img_height: int = 40, is_train: bool = True):
        self.samples = samples
        self.char2idx = char2idx
        self.img_width = img_width
        self.img_height = img_height
        self.is_train = is_train

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        # 1. Đọc và chuyển sang mức xám
        try:
            img = Image.open(img_path).convert("L")
        except Exception:
            img = Image.new("L", (self.img_width, self.img_height), color=255)

        # 2. Resize chuẩn (120, 40)
        img = img.resize((self.img_width, self.img_height), Image.BILINEAR)

        # 3. Chuẩn hóa Tensor [0, 1]
        img_np = np.array(img, dtype=np.float32) / 255.0
        img_tensor = torch.from_numpy(img_np).unsqueeze(0)  # (1, 40, 120)

        # 4. Mã hóa chuỗi nhãn thành tensor ID
        target = [self.char2idx[c] for c in label if c in self.char2idx]
        target_tensor = torch.tensor(target, dtype=torch.long)

        return img_tensor, target_tensor, label

def ctc_collate_fn(batch):
    """
    Collate function gộp batch cho CTC Loss:
    - images: (B, 1, 40, 120)
    - targets: 1D Tensor nối tiếp tất cả target ID của toàn bộ batch
    - input_lengths: Tensor độ dài chuỗi time-steps đầu ra của CNN (29)
    - target_lengths: Tensor độ dài nhãn từng mẫu
    - raw_labels: Danh sách nhãn gốc để kiểm tra độ chính xác
    """
    images, targets, raw_labels = zip(*batch)
    images = torch.stack(images, dim=0)

    target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.long)
    targets_concat = torch.cat(targets, dim=0)

    batch_size = images.size(0)
    # CNN Backbone giảm chiều rộng từ 120 -> 29 time-steps
    input_lengths = torch.full(size=(batch_size,), fill_value=29, dtype=torch.long)

    return images, targets_concat, input_lengths, target_lengths, raw_labels

# Khởi tạo DataLoader
BATCH_SIZE = 64
train_loader = DataLoader(CaptchaDataset(train_samples, char2idx, is_train=True), batch_size=BATCH_SIZE, shuffle=True, collate_fn=ctc_collate_fn)
val_loader = DataLoader(CaptchaDataset(val_samples, char2idx, is_train=False), batch_size=BATCH_SIZE, shuffle=False, collate_fn=ctc_collate_fn)
test_loader = DataLoader(CaptchaDataset(test_samples, char2idx, is_train=False), batch_size=BATCH_SIZE, shuffle=False, collate_fn=ctc_collate_fn)

print(f"[*] Số batch mỗi epoch: Train={len(train_loader)}, Val={len(val_loader)}, Test={len(test_loader)}")

## Phần 4: Trực Quan Hóa Một Số Mẫu Dữ Liệu

In [ ]:
# Lấy 1 batch từ DataLoader và hiển thị ảnh mẫu
sample_images, _, _, _, sample_labels = next(iter(train_loader))

plt.figure(figsize=(12, 4))
for i in range(min(6, len(sample_images))):
    plt.subplot(2, 3, i + 1)
    img_disp = sample_images[i].squeeze(0).numpy()
    plt.imshow(img_disp, cmap="gray")
    plt.title(f"Label: '{sample_labels[i]}'")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Phần 5: Thiết Kế Kiến Trúc Mô Hình CRNN (CNN + BiLSTM + CTC)

In [ ]:
class CRNN(nn.Module):
    """
    Kiến trúc mạng CRNN (Convolutional Recurrent Neural Network):
    1. CNN Backbone: 5 tầng Conv2D + BatchNorm + ReLU trích xuất đặc trưng hình ảnh.
    2. BiLSTM: 2 tầng LSTM 2 chiều học quan hệ chuỗi ký tự theo chiều ngang.
    3. Linear Projection: Chiếu sang phân phối xác suất từ vựng (num_classes).
    """
    def __init__(self, num_classes: int, hidden_size: int = 128):
        super(CRNN, self).__init__()
        self.num_classes = num_classes

        # 1. CNN Feature Extractor
        self.cnn = nn.Sequential(
            # Block 1: (B, 1, 40, 120) -> (B, 32, 20, 60)
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 2: (B, 32, 20, 60) -> (B, 64, 10, 30)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),

            # Block 3: (B, 64, 10, 30) -> (B, 128, 5, 30)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),

            # Block 4: (B, 128, 5, 30) -> (B, 256, 2, 30)
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),

            # Block 5: (B, 256, 2, 30) -> (B, 256, 1, 29)
            nn.Conv2d(256, 256, kernel_size=2, padding=0),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        # 2. Sequence Modeling (Bidirectional LSTM)
        self.rnn = nn.LSTM(
            input_size=256,
            hidden_size=hidden_size,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.2
        )

        # 3. CTC Classifier Head
        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        # x: (B, 1, H=40, W=120)
        features = self.cnn(x)                # (B, 256, 1, 29)
        features = features.squeeze(2)        # (B, 256, 29)
        features = features.permute(0, 2, 1)  # (B, 29, 256) (Batch, Time_Steps, Channels)

        rnn_out, _ = self.rnn(features)       # (B, 29, hidden_size*2)
        logits = self.fc(rnn_out)             # (B, 29, num_classes)
        log_probs = F.log_softmax(logits, dim=-1)
        
        return log_probs

def decode_greedy(log_probs, idx2char: dict, blank_idx: int = 0) -> list[str]:
    """
    Giải mã CTC Greedy Search: Lấy argmax tại mỗi time-step, gộp các ký tự lặp liên tiếp và bỏ token blank (0)
    """
    preds = torch.argmax(log_probs, dim=-1)  # (B, Time_Steps)
    decoded_texts = []
    for seq in preds:
        tokens = []
        prev_idx = -1
        for idx_val in seq:
            idx = idx_val.item() if isinstance(idx_val, torch.Tensor) else idx_val
            if idx != blank_idx and idx != prev_idx:
                tokens.append(idx2char.get(idx, ""))
            prev_idx = idx
        decoded_texts.append("".join(tokens))
    return decoded_texts

# Khởi tạo mô hình
model = CRNN(num_classes=num_classes, hidden_size=128).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[*] Khởi tạo mô hình CRNN thành công!")
print(f"[*] Tổng số tham số có thể huấn luyện (Trainable Parameters): {total_params:,}")

## Phần 6: Hàm Huấn Luyện & Đánh Giá (Training & Evaluation Pipeline)

In [ ]:
def evaluate(model, val_loader, criterion, idx2char, device):
    """
    Đánh giá mô hình trên tập Validation hoặc Test
    """
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_gts = []

    with torch.no_grad():
        for images, targets, input_lengths, target_lengths, raw_labels in val_loader:
            images = images.to(device)
            targets = targets.to(device)
            input_lengths = input_lengths.to(device)
            target_lengths = target_lengths.to(device)

            log_probs = model(images)  # (B, T, num_classes)
            log_probs_perm = log_probs.permute(1, 0, 2)  # (T, B, num_classes) cho CTCLoss
            loss = criterion(log_probs_perm, targets, input_lengths, target_lengths)
            val_loss += loss.item() * images.size(0)

            decoded = decode_greedy(log_probs, idx2char)
            all_preds.extend(decoded)
            all_gts.extend(raw_labels)

    avg_loss = val_loss / len(val_loader.dataset)
    
    # Tính độ chính xác toàn chuỗi (Exact Match)
    exact_matches = sum(1 for p, g in zip(all_preds, all_gts) if p == g)
    exact_acc = (exact_matches / len(all_gts)) * 100 if all_gts else 0.0

    return avg_loss, exact_acc, all_preds, all_gts

# Thiết lập Loss, Optimizer và Scheduler
criterion = nn.CTCLoss(blank=0, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

EPOCHS = 12
best_val_acc = 0.0
history = {"train_loss": [], "val_loss": [], "val_acc": []}
CHECKPOINT_PATH = os.path.join(BASE_DIR, "best_crnn_model.pth")

print("=" * 70)
print("                BẮT ĐẦU HUẤN LUYỆN MÔ HÌNH CRNN-CTC")
print("=" * 70)

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    model.train()
    train_loss = 0.0

    for images, targets, input_lengths, target_lengths, _ in train_loader:
        images = images.to(device)
        targets = targets.to(device)
        input_lengths = input_lengths.to(device)
        target_lengths = target_lengths.to(device)

        optimizer.zero_grad()
        log_probs = model(images)
        log_probs_perm = log_probs.permute(1, 0, 2)

        loss = criterion(log_probs_perm, targets, input_lengths, target_lengths)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss = train_loss / len(train_samples)
    val_loss, val_exact_acc, _, _ = evaluate(model, val_loader, criterion, idx2char, device)
    scheduler.step(val_exact_acc)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_exact_acc)

    elapsed = time.time() - start_time
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({elapsed:.1f}s) | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Exact Match Acc: {val_exact_acc:.2f}%")

    if val_exact_acc > best_val_acc:
        best_val_acc = val_exact_acc
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_acc": val_exact_acc,
            "char2idx": char2idx,
            "idx2char": idx2char,
            "num_classes": num_classes
        }, CHECKPOINT_PATH)
        print(f"  --> [*] Đã lưu mô hình tốt nhất ({val_exact_acc:.2f}%) tại: {CHECKPOINT_PATH}")

## Phần 7: Biểu Đồ Huấn Luyện & Đánh Giá Trên Tập Test

In [ ]:
# 1. Vẽ đồ thị Loss và Accuracy
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, len(history["train_loss"]) + 1), history["train_loss"], label="Train Loss", marker="o")
plt.plot(range(1, len(history["val_loss"]) + 1), history["val_loss"], label="Val Loss", marker="s")
plt.title("Loss Curves")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(history["val_acc"]) + 1), history["val_acc"], label="Val Exact Match Acc (%)", color="green", marker="o")
plt.title("Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# 2. Tải Checkpoint tốt nhất và Đánh giá trên tập Test
if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"[*] Đã tải Checkpoint tốt nhất từ Epoch {ckpt['epoch']} (Val Acc: {ckpt['val_acc']:.2f}%)")

test_loss, test_exact_acc, test_preds, test_gts = evaluate(model, test_loader, criterion, idx2char, device)
print("\n" + "=" * 50)
print(f"  KẾT QUẢ TRÊN TẬP KIỂM THỬ (TEST SET)")
print("=" * 50)
print(f"• Test Loss:              {test_loss:.4f}")
print(f"• Test Exact Match Acc:   {test_exact_acc:.2f}%")
print("=" * 50)

# Hiển thị 10 mẫu dự đoán thực tế
print("\n--- 10 Mẫu dự đoán ngẫu nhiên trên tập Test ---")
for i in range(min(10, len(test_preds))):
    status = "[V] Đúng" if test_preds[i] == test_gts[i] else "[X] Sai"
    print(f"  {status:<8} | Ground Truth: '{test_gts[i]:<6}' | Dự đoán: '{test_preds[i]}'")

## Phần 8: Xuất Mô Hình Sang Định Dạng ONNX & Đo Lường Tốc Độ Suy Luận

In [ ]:
# Đường dẫn xuất file ONNX
ONNX_OUTPUT_NOTEBOOK = os.path.join(BASE_DIR, "huit_captcha_crnn.onnx")
ONNX_OUTPUT_MODELS = os.path.abspath(os.path.join(BASE_DIR, "..", "models", "huit_captcha_crnn.onnx"))
os.makedirs(os.path.dirname(ONNX_OUTPUT_MODELS), exist_ok=True)

model.eval()
model.to("cpu")
dummy_input = torch.randn(1, 1, 40, 120, device="cpu")

print(f"[*] Đang xuất mô hình sang định dạng ONNX...")
for onnx_target in [ONNX_OUTPUT_NOTEBOOK, ONNX_OUTPUT_MODELS]:
    torch.onnx.export(
        model,
        dummy_input,
        onnx_target,
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=["input"],
        output_names=["output"],
        dynamic_axes={
            "input": {0: "batch_size"},
            "output": {0: "batch_size"}
        }
    )
    file_size_mb = os.path.getsize(onnx_target) / (1024 * 1024)
    print(f"  [+] Đã lưu file ONNX: {onnx_target} ({file_size_mb:.2f} MB)")

# Kiểm tra tính hợp lệ bằng onnx.checker
onnx_model = onnx.load(ONNX_OUTPUT_NOTEBOOK)
onnx.checker.check_model(onnx_model)
print("[+] onnx.checker: Mô hình ONNX hợp lệ chuẩn 100%!")

# Khởi tạo ONNX Runtime Inference Session
ort_session = ort.InferenceSession(ONNX_OUTPUT_NOTEBOOK, providers=["CPUExecutionProvider"])

# Đo lường thời gian suy luận (Latency Benchmark)
latencies = []
for _ in range(100):
    dummy_np = np.random.randn(1, 1, 40, 120).astype(np.float32)
    t0 = time.perf_counter()
    ort_session.run(None, {"input": dummy_np})
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000)

avg_latency = np.mean(latencies[10:])  # Bỏ qua 10 lượt warm-up đầu
p95_latency = np.percentile(latencies[10:], 95)

print("\n" + "=" * 50)
print("          BENCHMARK HIỆU NĂNG ONNX RUNTIME")
print("=" * 50)
print(f"• Thời gian suy luận trung bình:  {avg_latency:.2f} ms/ảnh")
print(f"• P95 Latency:                     {p95_latency:.2f} ms/ảnh")
print(f"• Tốc độ xử lý tương đương:       {1000/avg_latency:.0f} ảnh/giây (CPU)")
print("=" * 50)